# Running simulations with CDK4/6i present


In [1]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import datetime
import pickle

sys.path.insert(0, "cellular_cycle")

from cell_cycle_death_model import model
from model_fxns import *
from pysb.simulator import ScipyOdeSimulator
from pysb.simulator import BngSimulator


# Where to save large simulation output files
SAVE_DIR = Path(Path.home(),'ECM_mod_output/cdk46i_sims')
SAVE_DIR.mkdir(exist_ok=True)

OVERWRITE = False # Overwrite existing simulation files
SET_JPARAM = False

CURDATE = datetime.datetime.now().strftime('%Y%m%d')

### Define time range for entire notebook

In [2]:
t = np.linspace(0, 20000, 200001)

In [3]:
# Importing the parameters:
new_param = pd.read_csv('model_data/parameters.csv')
for key in new_param.keys():
    try:
        if model.parameters[key].value != new_param[key][0]:
            print(f'Changing {key} from {model.parameters[key].value} to {new_param[key][0]}')
            model.parameters[key].value = new_param[key]
    except:
        print(f'{key} not found in model parameters')

In [4]:
if SET_JPARAM:
    with open('fit_data/pso/frac_prolif_parameters_pso.pkl', 'rb') as f:
        frac_prolif_parameters_pso = pickle.load(f)
    new_param_pso_sort = frac_prolif_parameters_pso.sort_values([('cost', 'total cost likelihood')])
    new_params = new_param_pso_sort['model parameters']
    # Importing the parameters:
    ii = 0
    for key in new_params.keys():
        try:
            if model.parameters[key].value != new_params[key].iloc[ii]:
                print(f'Changing {key} from {model.parameters[key].value} to {new_param[key][0]}')
                model.parameters[key].value = new_params[key].iloc[ii]
        except:
            print(f'{key} not found in model parameters')

In [5]:
ecm_conditions = pd.read_csv('ECM_conditions.csv', index_col=[0])
ecm_conditions

,stiffness,drug,species,value
condition,,,,
stiff_DMSO,stiff,DMSO,AKT,1.0
stiff_DMSO,stiff,DMSO,ERK,1.0
stiff_DMSO,stiff,DMSO,FAK,1.0
int_DMSO,intermediate,DMSO,AKT,1.0
int_DMSO,intermediate,DMSO,ERK,1.0
int_DMSO,intermediate,DMSO,FAK,0.6
soft_DMSO,soft,DMSO,AKT,0.6
soft_DMSO,soft,DMSO,ERK,1.0
soft_DMSO,soft,DMSO,FAK,0.4


In [6]:
conditions = ecm_conditions.index.unique().tolist()
conditions


['stiff_DMSO', 'int_DMSO', 'soft_DMSO', 'stiff_erl', 'int_erl', 'soft_erl']

### Set CDK4/6i value
Will use each value in a set.

In [7]:
cdk46i_concs = [0.001, 0.01, 0.02, 0.05, 0.1, 0.2, 0.4, 1.0, 2.0, 4.0,] # values in µM

### ODE simulations

In [8]:
for cycdi_val in cdk46i_concs:
    model.parameters['CycDi_0'].value = cycdi_val
    for condition in ['stiff_DMSO', 'stiff_erl']:
        input_vals = pd.Series(ecm_conditions.loc[condition]['value'].to_list())
        input_vals.index = ecm_conditions.loc[condition]['species'].to_list()
        model.parameters['ERK'].value = input_vals['ERK']
        model.parameters['FAK'].value = input_vals['FAK']
        model.parameters['AKT'].value = input_vals['AKT']

        print(f"Running ODE simulation for {condition} and CDK4/6i conc of {cycdi_val}...")
    
        simres = ScipyOdeSimulator(model, t, verbose=False).run(); #deterministic simulation
        yout = simres.observables

        cycdi = str(cycdi_val).replace('.','_')
        
        if SET_JPARAM:
            filename = Path(SAVE_DIR, f'ODE_sim_{condition}_psoparam_cycdi_{cycdi}.csv')
        else:
            filename = Path(SAVE_DIR, f'ODE_sim_{condition}_param_cycdi_{cycdi}.csv')
        if OVERWRITE or not os.path.exists(filename):
            print(f"Saving to {filename}...")
            np.savetxt(filename, yout, delimiter=',')
        else:
            print(f"File {filename} already exists. Skipping...")

Running ODE simulation for stiff_DMSO and CDK4/6i conc of 0.001...


ld: warning: duplicate -rpath '/Users/darren/miniforge3/envs/ecm_model/lib' ignored


Saving to /Users/darren/ECM_mod_output/cdk46i_sims/ODE_sim_stiff_DMSO_param_cycdi_0_001.csv...
Running ODE simulation for stiff_erl and CDK4/6i conc of 0.001...
Saving to /Users/darren/ECM_mod_output/cdk46i_sims/ODE_sim_stiff_erl_param_cycdi_0_001.csv...
Running ODE simulation for stiff_DMSO and CDK4/6i conc of 0.01...
Saving to /Users/darren/ECM_mod_output/cdk46i_sims/ODE_sim_stiff_DMSO_param_cycdi_0_01.csv...
Running ODE simulation for stiff_erl and CDK4/6i conc of 0.01...
Saving to /Users/darren/ECM_mod_output/cdk46i_sims/ODE_sim_stiff_erl_param_cycdi_0_01.csv...
Running ODE simulation for stiff_DMSO and CDK4/6i conc of 0.02...
Saving to /Users/darren/ECM_mod_output/cdk46i_sims/ODE_sim_stiff_DMSO_param_cycdi_0_02.csv...
Running ODE simulation for stiff_erl and CDK4/6i conc of 0.02...
Saving to /Users/darren/ECM_mod_output/cdk46i_sims/ODE_sim_stiff_erl_param_cycdi_0_02.csv...
Running ODE simulation for stiff_DMSO and CDK4/6i conc of 0.05...
Saving to /Users/darren/ECM_mod_output/cdk4

### Stochastic simulations

In [9]:
for cycdi_val in cdk46i_concs:
    model.parameters['CycDi_0'].value = cycdi_val
    for condition in ['stiff_DMSO', 'stiff_erl']:
        input_vals = pd.Series(ecm_conditions.loc[condition]['value'].to_list())
        input_vals.index = ecm_conditions.loc[condition]['species'].to_list()
        model.parameters['ERK'].value = input_vals['ERK']
        model.parameters['FAK'].value = input_vals['FAK']
        model.parameters['AKT'].value = input_vals['AKT']

        print(f"Running stochastic simulation for {condition} and CDK4/6i conc of {cycdi_val} ...")

        sim = BngSimulator(model, tspan=t, verbose=False); #stochastic simulator
        simres = sim.run(method='ssa', verbose=False, seed=33); #running the simulation; remove seed for truly random; seed=33 is used for similarity to deterministic sim
        yout = simres.observables;

        cycdi = str(cycdi_val).replace('.','_')
        
        if SET_JPARAM:
            filename = Path(SAVE_DIR, f'stoch_sim_{condition}_psoparam_cycdi_{cycdi}.csv')
        else:
            filename = Path(SAVE_DIR, f'stoch_sim_{condition}_param_cycdi_{cycdi}.csv')
        if OVERWRITE or not os.path.exists(filename):
            print(f"Saving to {filename}...")
            np.savetxt(filename, yout, delimiter=',')
        else:
            print(f"File {filename} already exists. Skipping...")

Running stochastic simulation for stiff_DMSO and CDK4/6i conc of 0.001 ...
Saving to /Users/darren/ECM_mod_output/cdk46i_sims/stoch_sim_stiff_DMSO_param_cycdi_0_001.csv...
Running stochastic simulation for stiff_erl and CDK4/6i conc of 0.001 ...
Saving to /Users/darren/ECM_mod_output/cdk46i_sims/stoch_sim_stiff_erl_param_cycdi_0_001.csv...
Running stochastic simulation for stiff_DMSO and CDK4/6i conc of 0.01 ...
Saving to /Users/darren/ECM_mod_output/cdk46i_sims/stoch_sim_stiff_DMSO_param_cycdi_0_01.csv...
Running stochastic simulation for stiff_erl and CDK4/6i conc of 0.01 ...
Saving to /Users/darren/ECM_mod_output/cdk46i_sims/stoch_sim_stiff_erl_param_cycdi_0_01.csv...
Running stochastic simulation for stiff_DMSO and CDK4/6i conc of 0.02 ...
Saving to /Users/darren/ECM_mod_output/cdk46i_sims/stoch_sim_stiff_DMSO_param_cycdi_0_02.csv...
Running stochastic simulation for stiff_erl and CDK4/6i conc of 0.02 ...
Saving to /Users/darren/ECM_mod_output/cdk46i_sims/stoch_sim_stiff_erl_param_